# Flash Flash Revolution Contested Difficulties - Bronze Layer Google Sheets Ingestion

This notebook automatically ingests difficulty adjustment data from a Google Sheets document using the Google Sheets API for reliable, automated sheet discovery and processing.

## Data Source

**Google Sheets URL:**
```
https://docs.google.com/spreadsheets/d/1Wm1RHG318EK07U4VDXkztKTKnfy9wEOQqWRiTclbCz8/edit?gid=0#gid=0
```

**Document Structure:**
- Multiple sheets, one per year (e.g., "2025 Difficulty Changes", "2024 Difficulty Changes")
- Header row: Song Name, Old Diff, New Diff, Date
- Contains historical contested difficulty assessments from community

---

## Automated Discovery Approach

**Why Google Sheets API:**
- Automatically discovers all sheets in the document
- No hardcoded sheet names or GIDs required
- Self-updating when new year sheets are added
- Reliable authentication via service account

**Discovery Logic:**
1. Connect to Google Sheets API with service account credentials
2. Fetch metadata for all sheets in the document
3. Filter for sheets containing "Difficulty Changes" in the name
4. Extract sheet GIDs for data fetching
5. Skip sheets not matching the filter (e.g., "Contested Difficulties" summary sheet)

**Authentication:**
- Service account credentials stored in Unity Catalog Volume
- Path: `/Volumes/acubed/ffr/assets/google-cloud-credentials.json`
- Read-only scope: `spreadsheets.readonly`

---

## Data Ingestion Process

**For Each Discovered Sheet:**
1. Construct CSV export URL using sheet GID
2. Fetch CSV data via Google Sheets export endpoint
3. Parse CSV with pandas (skip empty first row)
4. Standardize column names (lowercase, replace spaces with underscores)
5. Add `source_sheet` column to track data origin
6. Convert to Spark DataFrame

**Column Name Standardization:**
- "Song Name" → "song_name"
- "Old Diff" → "old_diff"
- "New Diff" → "new_diff"
- "Date" → "date"

---

## Output Table

**Table:** `acubed.ffr.bronze__contested-difficulties`

**Schema:**
- `song_name` (string): Name of the contested song
- `old_diff` (string): Previous difficulty value (raw string from sheet)
- `new_diff` (string): Proposed new difficulty value (raw string from sheet)
- `date` (string): Date of difficulty change proposal (raw string from sheet)
- `source_sheet` (string): Origin sheet name (e.g., "2025 Difficulty Changes")
- `ingestion_timestamp` (timestamp): When data was loaded into bronze layer

**Write Mode:**
- Full overwrite on each run
- Combines all discovered sheets into single table
- Preserves source_sheet for lineage tracking

---

## Data Characteristics

**Raw String Values:**
- All columns stored as strings (no parsing at bronze layer)
- Date formats vary by year/entry (transformation handled in silver layer)
- `old_diff` and `new_diff` contain special notations:
  - "no change" - No difficulty adjustment
  - "change" - Adjustment occurred but value not recorded
  - "(resolved) X" - Marked as resolved with final value X
  - Numeric strings - Standard difficulty values

**Multi-Year Coverage:**
- Sheets span multiple years of difficulty adjustments
- Each year may have different data quality and format conventions
- Silver layer transformation normalizes these variations

In [0]:
%%capture
%pip install google-auth google-auth-oauthlib google-api-python-client

dbutils.library.restartPython()

In [0]:
import pandas as pd
import requests
from pyspark.sql.functions import col, current_timestamp

print("✓ Libraries imported successfully")

In [0]:
# Use Google Sheets API for automatic sheet discovery
# Only ingests sheets with "Difficulty Changes" in the name

from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials

print("Discovering sheets via Google Sheets API...")
print("=" * 70)

# Path to service account credentials in Unity Catalog volume
credentials_path = "/Volumes/acubed/ffr/assets/google-cloud-credentials.json"

# Authenticate
SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly"]
creds = Credentials.from_service_account_file(credentials_path, scopes=SCOPES)

service = build("sheets", "v4", credentials=creds)

# Fetch spreadsheet metadata
sheet_id = "1Wm1RHG318EK07U4VDXkztKTKnfy9wEOQqWRiTclbCz8"
sheet_metadata = service.spreadsheets().get(spreadsheetId=sheet_id).execute()

# Extract sheets with "Difficulty Changes" in the name
api_discovered_sheets = {}

print(f"✓ Connected to Google Sheets API")
print(f"✓ Found {len(sheet_metadata['sheets'])} total sheet(s)")
print(f"\nFiltering for sheets containing 'Difficulty Changes'...\n")

for sheet in sheet_metadata["sheets"]:
    sheet_title = sheet["properties"]["title"]
    sheet_gid = str(sheet["properties"]["sheetId"])
    
    # Only include sheets with "Difficulty Changes" in the name
    if "Difficulty Changes" in sheet_title:
        api_discovered_sheets[sheet_gid] = sheet_title
        print(f"  ✓ {sheet_title} (gid={sheet_gid})")
    else:
        print(f"  ⊘ {sheet_title} (gid={sheet_gid}) - skipped")

print("\n" + "=" * 70)
print(f"✓ Discovered {len(api_discovered_sheets)} sheets matching filter")
print("=" * 70)

In [0]:
import pandas as pd
import requests

# Base Google Sheets URL
sheet_id = "1Wm1RHG318EK07U4VDXkztKTKnfy9wEOQqWRiTclbCz8"
base_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}"

print("Fetching data from discovered sheets...")
print("=" * 70)

all_sheets_data = {}

for gid, sheet_name in api_discovered_sheets.items():
    csv_url = f"{base_url}/export?format=csv&gid={gid}"
    print(f"\n→ Fetching: {sheet_name} (gid={gid})")
    
    try:
        # Read CSV - Google Sheets exports have an empty first row, 
        # then headers in row 1, then data starting from row 2
        df_raw = pd.read_csv(csv_url, header=None)
        
        # Use second row (index 1) as column names
        headers = df_raw.iloc[1].fillna('').astype(str).tolist()
        
        # Take data starting from row 2 (skip empty row 0 and header row 1)
        df_pandas = df_raw[2:].copy()
        df_pandas.columns = headers
        
        # Drop any completely empty columns
        df_pandas = df_pandas.loc[:, df_pandas.columns != '']
        
        # Reset index
        df_pandas = df_pandas.reset_index(drop=True)
        
        all_sheets_data[sheet_name] = df_pandas
        print(f"  ✓ {len(df_pandas):,} rows, {len(df_pandas.columns)} columns")
        print(f"  ✓ Headers: {', '.join(df_pandas.columns.tolist()[:5])}..." if len(df_pandas.columns) > 5 else f"  ✓ Headers: {', '.join(df_pandas.columns.tolist())}")
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 70)
print(f"✓ Successfully fetched {len(all_sheets_data)} sheets")
print("=" * 70)

In [0]:
print("Analyzing all sheets...")
print("=" * 70)

for sheet_name, df_pandas in all_sheets_data.items():
    print(f"\n{'=' * 70}")
    print(f"SHEET: {sheet_name}")
    print("=" * 70)
    
    # Convert to Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    print(f"\nRows: {df_spark.count():,}")
    print(f"Columns: {len(df_spark.columns)}")
    
    print("\nSchema:")
    df_spark.printSchema()
    
    print("\nSample data (first 10 rows):")
    display(df_spark.limit(10))
    
    # Store Spark DataFrames for later use
    if 'all_sheets_spark' not in globals():
        all_sheets_spark = {}
    all_sheets_spark[sheet_name] = df_spark

print("\n" + "=" * 70)
print(f"✓ Analysis complete for {len(all_sheets_data)} sheets")
print("=" * 70)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
import re

print("Combining all sheets into bronze__contested-difficulties...")
print("=" * 70)

# Combine all sheets into a single DataFrame
combined_dfs = []

for sheet_name, df_pandas in all_sheets_data.items():
    print(f"\n→ Processing sheet: {sheet_name}")
    print(f"   Rows: {len(df_pandas):,}, Columns: {len(df_pandas.columns)}")
    
    # Convert to Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    # Clean column names: lowercase, replace spaces with underscores, remove special chars
    print(f"   Cleaning column names...")
    for old_col in df_spark.columns:
        # Create SQL-friendly column name
        new_col = old_col.lower()  # Lowercase
        new_col = new_col.replace(' ', '_')  # Replace spaces with underscores
        new_col = re.sub(r'[^a-z0-9_]', '', new_col)  # Remove special characters
        
        if old_col != new_col:
            df_spark = df_spark.withColumnRenamed(old_col, new_col)
            print(f"     • {old_col} → {new_col}")
    
    # Add source sheet identifier
    df_with_source = df_spark.withColumn("source_sheet", lit(sheet_name))
    
    combined_dfs.append(df_with_source)
    print(f"   ✓ Added to combined dataset")

print("\n" + "=" * 70)
print("Unioning all sheets...")
print("=" * 70)

# Union all DataFrames (using unionByName to handle different schemas if needed)
from functools import reduce
from pyspark.sql import DataFrame

if len(combined_dfs) == 1:
    df_combined = combined_dfs[0]
else:
    df_combined = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), combined_dfs)

# Add ingestion timestamp
df_final = df_combined.withColumn("ingestion_timestamp", current_timestamp())

total_rows = df_final.count()
print(f"Total rows across all sheets: {total_rows:,}")

# Save as bronze__contested-difficulties
table_name = "acubed.ffr.`bronze__contested-difficulties`"
print(f"\nSaving to {table_name}...")

df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"\n✓ Created {table_name}")
print(f"  Rows: {total_rows:,}")
print(f"  Sheets combined: {len(all_sheets_data)}")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
for sheet_name in all_sheets_data.keys():
    print(f"  • {sheet_name}")

print(f"\nAll {len(all_sheets_data)} sheets combined into: {table_name}")
print(f"\nFinal schema:")
df_final.printSchema()
print("=" * 70)